In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision import transforms
from torchvision.models import resnet34
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from thop import profile

In [2]:
class MiniImageNetDataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []
        
        # 讀取txt文件中的路徑和標籤
        with open(txt_file, 'r') as f:
            for line in f:
                img_path, label = line.strip().split()
                self.data.append((img_path, int(label)))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        full_path = os.path.join(self.root_dir, img_path)
        
        img = Image.open(full_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

def load_datasets(root_dir, txt_dir, train_transform, val_transform, batch_size):
    train_dataset = MiniImageNetDataset(os.path.join(txt_dir, 'train.txt'), root_dir, train_transform)
    val_dataset = MiniImageNetDataset(os.path.join(txt_dir, 'val.txt'), root_dir, val_transform)
    test_dataset = MiniImageNetDataset(os.path.join(txt_dir, 'test.txt'), root_dir, val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

def get_transforms():
    # 訓練集的轉換
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),  # ResNet34預設輸入尺寸為224x224
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 驗證集的轉換
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform


In [3]:
class FourLayerWideNetwork(nn.Module):
    def __init__(self, num_classes=100, width_factor=1):
        super(FourLayerWideNetwork, self).__init__()

        c1 = 32  * width_factor
        c2 = 64  * width_factor
        c3 = 128 * width_factor
        c4 = 256 * width_factor

        self.features = nn.Sequential(
            # Layer1
            nn.Conv2d(3,  c1, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(c1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            # Layer2
            nn.Conv2d(c1, c2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c2),
            nn.GELU(),
            nn.Dropout(0.1),

            # Layer3
            nn.Conv2d(c2, c3, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(c3),
            nn.GELU(),
            nn.Dropout(0.1),

            # Layer4
            nn.Conv2d(c3, c4, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(c4),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256 * width_factor, num_classes),
        )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
def evaluate_model(model, model_path, dataloader, device):
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    correct = 0
    total = 0
    print('start evaluate...')
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    return accuracy



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'using device : {device}')

batch_size = 32
root_dir = 'dataset'
txt_dir = 'dataset'

# 載入資料
print('loading data...')
train_transform, val_transform = get_transforms()
train_loader, val_loader, test_loader = load_datasets(root_dir, txt_dir, train_transform, val_transform, batch_size)

# 初始化模型
print('initial model...')
my_model = FourLayerWideNetwork(num_classes=100, width_factor=6)
resnet_model = resnet34(num_classes=100)

# 模型路徑
my_path = 'Taskb_checkpoints/wide_four/best_model.pth'
resnet_path = 'Taskb_checkpoints/resnet_34/best_model.pth'

# 評估準確率

my_train_acc = evaluate_model(my_model, my_path, train_loader, device)
my_val_acc   = evaluate_model(my_model, my_path, val_loader, device)
my_test_acc  = evaluate_model(my_model, my_path, test_loader, device)

resnet_train_acc = evaluate_model(resnet_model, resnet_path, train_loader, device)
resnet_val_acc   = evaluate_model(resnet_model, resnet_path, val_loader, device)
resnet_test_acc  = evaluate_model(resnet_model, resnet_path, test_loader, device)

print(f" Your Model - Train: {my_train_acc:.4f}, Val: {my_val_acc:.4f}, Test: {my_test_acc:.4f}")
print(f" ResNet34   - Train: {resnet_train_acc:.4f}, Val: {resnet_val_acc:.4f}, Test: {resnet_test_acc:.4f}")


using device : cuda
loading data...
initial model...


/tmp/ipykernel_4174931/1028123388.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


 Your Model - Train: 0.4783, Val: 0.5889, Test: 0.5867
 ResNet34   - Train: 0.5573, Val: 0.6511, Test: 0.6711


---

In [7]:
# 計算模型參數量
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [14]:
# 三個模型的寬度
widths = [2, 4, 6]
results = []

for w in widths:
    # 建立模型實例
    model = FourLayerWideNetwork(num_classes=100, width_factor=w)
    model_path = f'Taskb_checkpoints/wide_four/best_model_w{w}.pth'
    
    # 1. 準確率（僅測試集）
    acc = evaluate_model(model, model_path, test_loader, device)
    
    # 2. 參數量
    params = count_parameters(model)
    
    # 3. FLOPS（輸入 1×3×224×224）
    model_cpu = FourLayerWideNetwork(num_classes=100, width_factor=w).cpu()
    dummy = torch.randn(1, 3, 224, 224)
    flops, _ = profile(model_cpu, inputs=(dummy,), verbose=False)
    
    # 4. 模型容量（檔案大小，Bytes）
    size_bytes = os.path.getsize(model_path)
    
    results.append({
        'model': f'w{w}',
        'accuracy': acc,
        'parameters': params,
        'flops': flops,
        'model_size_bytes': size_bytes
    })

# 轉成 DataFrame 並存 CSV
df = pd.DataFrame(results)
csv_path = 'Taskb_checkpoints/wide_four/ablation_results.csv'
df.to_csv(csv_path, index=False)
print(f'➜ 結果已存為 {csv_path}')

# 指標清單
metrics = ['accuracy', 'parameters', 'flops', 'model_size_bytes']

for metric in metrics:
    plt.figure(figsize=(6, 4))
    values = df[metric]
    bars = plt.bar(df['model'], values, width=0.6)
    
    # 在每根 bar 上標註數值
    for bar in bars:
        h = bar.get_height()
        if metric == 'accuracy':
            label = f'{h:.4f}'
        else:
            label = f'{int(h):,}'
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            h,
            label,
            ha='center',
            va='bottom',
            fontsize=10
        )
    
    plt.title(f'Ablation Study – {metric}')
    plt.xlabel('Model')
    plt.ylabel(metric)
    plt.tight_layout()
    
    # 存檔
    save_path = f'Taskb_checkpoints/wide_four/ablation_{metric}.png'
    plt.savefig(save_path)
    plt.close()
    print(f'➜ 已存 {metric} 圖到 {save_path}')



/tmp/ipykernel_4174931/1028123388.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


➜ 結果已存為 Taskb_checkpoints/wide_four/ablation_results.csv
➜ 已存 accuracy 圖到 Taskb_checkpoints/wide_four/ablation_accuracy.png
➜ 已存 parameters 圖到 Taskb_checkpoints/wide_four/ablation_parameters.png
➜ 已存 flops 圖到 Taskb_checkpoints/wide_four/ablation_flops.png
➜ 已存 model_size_bytes 圖到 Taskb_checkpoints/wide_four/ablation_model_size_bytes.png
